In [ ]:
from importlib.util import find_spec
from pathlib import Path
import json
import os

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "faiss": "faiss-cpu",
    "sentence_transformers": "sentence-transformers",
}

missing = [f"{module} (pip install {package})" for module, package in REQUIRED_PACKAGES.items() if find_spec(module) is None]
if missing:
    raise ModuleNotFoundError(
        "Thiếu dependency để chạy retrieval eval bằng notebook chính thức: " + ", ".join(missing)
    )

import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

def find_ai_lab_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current] + list(current.parents):
        if candidate.name == "ai_lab":
            return candidate
    raise RuntimeError("Không tìm thấy thư mục ai_lab.")

AI_LAB_ROOT = find_ai_lab_root(Path.cwd())

KB_VERSION = os.getenv("HOMELAB_KB_VERSION", "v1")
RETRIEVER_VERSION = os.getenv("HOMELAB_RETRIEVER_VERSION", KB_VERSION)
DEFAULT_EVAL_SET_NAME = "health_rag_eval_v1_1.json" if KB_VERSION == "v1" else f"health_rag_eval_{KB_VERSION}_release_candidate.json"
DEFAULT_EVAL_REPORT_NAME = "retrieval_eval_v1_1.csv" if KB_VERSION == "v1" else f"retrieval_eval_{KB_VERSION}.csv"
EVAL_SET_NAME = os.getenv("HOMELAB_EVAL_SET_NAME", DEFAULT_EVAL_SET_NAME)
EVAL_REPORT_NAME = os.getenv("HOMELAB_EVAL_REPORT_NAME", DEFAULT_EVAL_REPORT_NAME)

ARTIFACTS_DIR = AI_LAB_ROOT / "artifacts" / f"retriever_{RETRIEVER_VERSION}"
DATASETS_DIR = AI_LAB_ROOT / "datasets"
REPORTS_DIR = AI_LAB_ROOT / "reports"

KB_CHUNKS_PATH = ARTIFACTS_DIR / f"kb_chunks_{KB_VERSION}.json"
FAISS_INDEX_PATH = ARTIFACTS_DIR / "faiss.index"
EMBEDDING_CONFIG_PATH = ARTIFACTS_DIR / "embedding_config.json"
EVAL_SET_PATH = DATASETS_DIR / "eval" / EVAL_SET_NAME
EVAL_REPORT_PATH = REPORTS_DIR / EVAL_REPORT_NAME

print("AI_LAB_ROOT =", AI_LAB_ROOT)
print("KB_CHUNKS_PATH =", KB_CHUNKS_PATH)
print("FAISS_INDEX_PATH =", FAISS_INDEX_PATH)
print("EMBEDDING_CONFIG_PATH =", EMBEDDING_CONFIG_PATH)
print("EVAL_SET_PATH =", EVAL_SET_PATH)
print("EVAL_REPORT_PATH =", EVAL_REPORT_PATH)


In [ ]:
with open(KB_CHUNKS_PATH, "r", encoding="utf-8") as f:
    kb_chunks = json.load(f)

with open(EMBEDDING_CONFIG_PATH, "r", encoding="utf-8") as f:
    embedding_config = json.load(f)

with open(EVAL_SET_PATH, "r", encoding="utf-8") as f:
    eval_set = json.load(f)

index = faiss.read_index(str(FAISS_INDEX_PATH))
model = SentenceTransformer(embedding_config["model_name"])

print("Chunks:", len(kb_chunks))
print("Eval queries:", len(eval_set))
print("Index total:", index.ntotal)

In [ ]:
def search(query, top_k=3):
    query_text = embedding_config["query_prefix"] + query.strip()
    q_emb = model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(q_emb, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        chunk = kb_chunks[idx]
        results.append({
            "score": float(score),
            "chunk_id": chunk["chunk_id"],
            "source_id": chunk["source_id"],
            "section": chunk["section"],
            "title": chunk["title"]
        })
    return results

In [ ]:
rows = []

for sample in eval_set:
    query = sample["query"]
    expected_chunk_ids = set(sample["expected_chunk_ids"])
    expected_source_id = sample["expected_source_id"]
    expected_section = sample["expected_section"]
    acceptable_source_ids = set(sample.get("acceptable_source_ids", [expected_source_id]))
    acceptable_sections = set(sample.get("acceptable_sections", [expected_section]))

    results = search(query, top_k=3)

    top1 = results[0] if len(results) > 0 else None
    top3_chunk_ids = [r["chunk_id"] for r in results[:3]]

    hit_at_1 = int(top1 is not None and top1["chunk_id"] in expected_chunk_ids)
    hit_at_3 = int(any(cid in expected_chunk_ids for cid in top3_chunk_ids))
    source_at_1 = int(top1 is not None and top1["source_id"] == expected_source_id)
    section_at_1 = int(top1 is not None and top1["section"] == expected_section)
    acceptable_source_at_1 = int(top1 is not None and top1["source_id"] in acceptable_source_ids)
    acceptable_section_at_1 = int(top1 is not None and top1["section"] in acceptable_sections)
    acceptable_top3_match = int(any((r["source_id"] in acceptable_source_ids and r["section"] in acceptable_sections) for r in results[:3]))

    rows.append({
        "query": query,
        "expected_chunk_ids": ",".join(expected_chunk_ids),
        "expected_source_id": expected_source_id,
        "expected_section": expected_section,
        "acceptable_source_ids": ",".join(sorted(acceptable_source_ids)),
        "acceptable_sections": ",".join(sorted(acceptable_sections)),
        "top1_chunk_id": top1["chunk_id"] if top1 else None,
        "top1_source_id": top1["source_id"] if top1 else None,
        "top1_section": top1["section"] if top1 else None,
        "top1_score": top1["score"] if top1 else None,
        "top3_chunk_ids": ",".join(top3_chunk_ids),
        "hit_at_1": hit_at_1,
        "hit_at_3": hit_at_3,
        "source_at_1": source_at_1,
        "section_at_1": section_at_1,
        "acceptable_source_at_1": acceptable_source_at_1,
        "acceptable_section_at_1": acceptable_section_at_1,
        "acceptable_top3_match": acceptable_top3_match,
    })

df_eval = pd.DataFrame(rows)
df_eval.to_csv(EVAL_REPORT_PATH, index=False, encoding="utf-8-sig")

print("Đã ghi:", EVAL_REPORT_PATH)
df_eval.head()


In [ ]:
metrics = {
    "Strict Recall@1": df_eval["hit_at_1"].mean(),
    "Strict Recall@3": df_eval["hit_at_3"].mean(),
    "Strict Source Accuracy@1": df_eval["source_at_1"].mean(),
    "Strict Section Accuracy@1": df_eval["section_at_1"].mean(),
    "Acceptable Source Accuracy@1": df_eval["acceptable_source_at_1"].mean(),
    "Acceptable Section Accuracy@1": df_eval["acceptable_section_at_1"].mean(),
    "Acceptable Top3 Match": df_eval["acceptable_top3_match"].mean(),
}

pd.DataFrame([metrics])


In [ ]:
wrong_top1 = df_eval[df_eval["hit_at_1"] == 0].copy()
wrong_top1

In [ ]:
wrong_top3 = df_eval[df_eval["hit_at_3"] == 0].copy()
wrong_top3